# Build a Minimal Agent: LangChain + OpenAI + Tools (Hands-on)

This notebook is a **fully runnable**, **fully annotated** demo that builds a minimal tool-using agent using:

- **LangChain** message abstractions + tool schemas
- **OpenAI chat model** (via `langchain-openai`)
- A **manual agent loop**: *Think → Act (tool) → Observe → Repeat → Answer*
- Real memory behavior via **`remember` + `recall`**
- **Structured tool inputs** (JSON schema arguments)

You will end with an agent that can:
- Answer normally
- Decide when to call tools
- Call tools with structured JSON arguments
- Use multiple tools in one request
- Store and recall notes from an internal memory store

---

## Prereqs
- You need an **OpenAI API key** available as environment variable: `OPENAI_API_KEY`.
- You need an **Tavily API key** available as environment variable: `TAVILY_API_KEY`.

## 1) Install dependencies

We install:
- `langchain`, `langchain-core` (messages/tools primitives)
- `langchain-openai` (OpenAI model wrapper)
- `python-dotenv` (optional `.env` loading)
- `requests` (For Tavily Search API Call)

> If you're on Colab, this works as-is.


### What this cell does
- Installs the Python dependencies needed for the demo.
- If you restart the runtime, you may need to re-run this cell.
- In Colab, installs can take a while if the runtime is reconnecting.


In [ ]:
!pip -q install -U langchain langchain-openai langchain-core python-dotenv requests


## 2) Configure API key

Preferred: set an environment variable before running:

- macOS/Linux: `export OPENAI_API_KEY="..."`
- Windows PowerShell: `$env:OPENAI_API_KEY="..."`

Alternative: set it in the notebook (not recommended if you'll share the notebook).


### What this cell does
- Loads API keys from **Google Colab Secrets** and places them into environment variables.
- LangChain/OpenAI reads keys from `os.environ`, so this must run **before** creating the model.
- We assert the keys exist to fail fast with a clear error.


In [ ]:
from google.colab import userdata
import os

# Option A (recommended): set OPENAI_API_KEY in your environment before running.
# Option B: uncomment and paste your key (not recommended for shared notebooks).
# os.environ["OPENAI_API_KEY"] = "YOUR_KEY_HERE"

os.environ["OPENAI_API_KEY"] = userdata.get("gpt_chat_key")
os.environ["TAVILY_API_KEY"] = userdata.get("tavily_api_key")
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set. Set it before running."
assert os.environ["TAVILY_API_KEY"], "TAVILY_API_KEY not found in Colab Secrets"

print("API_KEYS detected ✅")


## 3) Imports

We import:
- `ChatOpenAI` as our LLM interface
- `@tool` decorator to define tools with schemas
- Message classes to build the agent state


### What this cell does
- Imports LangChain primitives:
  - `ChatOpenAI` (LLM wrapper)
  - `@tool` (to expose Python functions as tools with schemas)
  - message types that form the agent state (`SystemMessage`, `HumanMessage`, `ToolMessage`).


In [ ]:
import math
import os
import requests
from typing import Any, Dict, List

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage


## 4) Define tools (with schemas)

Tools are just Python functions decorated with `@tool`.

**Structured tool inputs**
- Notice `calculator(expression: str, precision: int)` and `web_search(query: str, top_k: int)`.
- LangChain uses the function signature + docstring to expose a JSON schema to the model.
- The model will call tools with **structured JSON arguments**.

We also define:
- `remember(note)` to store a note into agent memory
- `recall(limit)` to retrieve saved notes


### What this cell does
- Defines the **tools** the agent can call.
- Each `@tool` function becomes a callable capability with a JSON schema derived from its signature.
- Includes:
  - `calculator(expression, precision)`: math evaluation with restricted globals.
  - `web_search(query, top_k)`: web search (Tavily in this notebook). The shared/public notebook can use a stub so it runs without extra keys.
  - `remember(note)` and `recall(limit)`: memory interface (storage handled in the agent loop).
- Registers tools in `TOOLS` and a name→tool lookup `TOOL_MAP`.


In [ ]:
# -----------------------------
# TOOL 1: Calculator (STRUCTURED INPUTS)
# -----------------------------
@tool
def calculator(expression: str, precision: int = 4) -> str:
    """
    Evaluate a basic math expression.

    Inputs:
    - expression: string like "2*(3+4)**2" or "math.sqrt(2)"
    - precision: integer number of decimals for floats

    Output:
    - result as a string
    """
    # Very small safety gate: allow only specific characters
    allowed = set("0123456789+-*/(). **abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ_")
    if any(ch not in allowed for ch in expression):
        return "Rejected: expression contains disallowed characters."

    try:
        # Remove builtins for safety; only expose math module
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        if isinstance(result, float):
            return str(round(result, precision))
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# -----------------------------
# TOOL 2: Web search (Real)
# -----------------------------
@tool
def web_search(query: str, top_k: int = 5) -> str:
    """
    Real web search using Tavily Search API.

    Inputs:
    - query: search query string
    - top_k: number of results to return (max 10 recommended)

    Output:
    - A compact text block with titles, urls, and short snippets
    """
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        return "Error: TAVILY_API_KEY is not set."

    top_k = max(1, min(int(top_k), 10))

    url = "https://api.tavily.com/search"
    payload = {
        "api_key": api_key,
        "query": query,
        "max_results": top_k,
        "search_depth": "basic",
        "include_answer": False,
        "include_raw_content": False,
    }

    try:
        r = requests.post(url, json=payload, timeout=30)
        r.raise_for_status()
        data = r.json()

        results = data.get("results", [])
        if not results:
            return "No results found."

        lines = []
        for i, item in enumerate(results, start=1):
            title = item.get("title", "").strip()
            link = item.get("url", "").strip()
            snippet = (item.get("content", "") or "").strip()
            snippet = snippet[:240] + ("..." if len(snippet) > 240 else "")

            lines.append(f"{i}. {title}\n   {link}\n   {snippet}")

        return "\n".join(lines)

    except requests.exceptions.RequestException as e:
        return f"Search error: {e}"


# -----------------------------
# TOOL 3: Remember (store note)
# -----------------------------
@tool
def remember(note: str) -> str:
    """Save a note into memory."""
    return f"Saved note: {note}"


# -----------------------------
# TOOL: Recall (retrieve notes)
# -----------------------------
@tool
def recall(limit: int = 10) -> str:
    """Return the most recent notes saved in memory."""
    # We'll override retrieval inside the agent loop (tools are stateless).
    return "Recall invoked (memory retrieval handled by agent)."

TOOLS = [calculator, web_search, remember, recall]
TOOL_MAP = {t.name: t for t in TOOLS}

print("Tools registered:", list(TOOL_MAP.keys()))


## 5) Create the model

We use `ChatOpenAI` and select a model.

> If your account doesn't have access to `gpt-4o-mini`, change it to another model you can use.


### What this cell does
- Creates the chat model wrapper (`ChatOpenAI`).
- Must run after the API key cell, otherwise you will get an error about missing `OPENAI_API_KEY`.


In [ ]:
def make_llm():
    return ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.2,
    )

llm = make_llm()
print("LLM ready ✅")


## 6) System prompt (agent policy)

This is the **agent constitution**:
- When to use tools
- How to format tool calls (JSON arguments)
- How to synthesize final answers
- How to use memory tools


### What this cell does
- Defines the **system prompt** (agent policy).
- This guides when to use tools, how to format tool calls, and how to write final answers.


In [ ]:
SYSTEM_PROMPT = """You are a minimal tool-using agent.

You have access to tools for:
- calculator(expression, precision)
- web_search(query, top_k)
- remember(note)
- recall(limit)

Rules:
1) If a tool can help, call it.
2) When calling a tool, provide valid JSON arguments matching the schema.
3) After tool results, decide whether to call more tools or answer.
4) Keep the final answer clean; do not mention tool calls unless asked.
5) Use remember/recall to store and retrieve user-provided info when relevant.
"""


## 7) The agent loop (the core)

This is the most important part.

The agent maintains:
- `self.messages` as **state** (system + user + tool results)
- `self.notes` as a tiny **memory store**

Algorithm:
1) Append user message
2) Ask the model what to do next (**with tools enabled**)
3) If the model calls tools:
   - execute tools
   - append tool outputs as `ToolMessage`
   - loop again
4) If no tool calls:
   - return final answer

Guardrail:
- `max_steps` prevents infinite tool loops.


### What this cell does
- Implements the **agent loop** (Think → Act → Observe → Repeat → Answer).
- Key ideas:
  - `self.messages` is state (system + user + tool observations).
  - `bind_tools(self.tools)` enables structured `tool_calls`.
  - `_execute_tool()` runs tools; results become `ToolMessage` observations.
  - `max_steps` prevents infinite loops.
- Memory behavior:
  - `remember` writes into `self.notes`.
  - `recall` reads from `self.notes`.


In [ ]:
class MinimalAgent:
    def __init__(self, llm: ChatOpenAI, system_prompt: str, tools: list):
        self.llm = llm
        self.tools = tools
        self.messages: List[Any] = [SystemMessage(content=system_prompt)]
        self.notes: List[str] = []
        self.max_steps = 8  # guardrail

    def _execute_tool(self, tool_call: Dict[str, Any]) -> str:
        tool_name = tool_call["name"]
        tool_args = tool_call.get("args", {}) or {}

        # ---- Real memory behavior ----
        if tool_name == "remember":
            note = tool_args.get("note", "")
            if note:
                self.notes.append(note)
            return str(TOOL_MAP[tool_name].invoke(tool_args))

        if tool_name == "recall":
            limit = int(tool_args.get("limit", 10))
            if not self.notes:
                return "No notes saved."
            recent = list(reversed(self.notes))[:max(1, limit)]
            return "\n".join([f"- {n}" for n in recent])

        # ---- Normal tools ----
        tool_fn = TOOL_MAP.get(tool_name)
        if tool_fn is None:
            return f"Unknown tool: {tool_name}"

        return str(tool_fn.invoke(tool_args))

    def run(self, user_input: str, verbose: bool = True) -> str:
        self.messages.append(HumanMessage(content=user_input))

        for step in range(self.max_steps):
            # Bind tool schemas so the model can emit tool_calls with JSON args
            response = self.llm.bind_tools(self.tools).invoke(self.messages)
            self.messages.append(response)

            tool_calls = getattr(response, "tool_calls", None) or []
            if not tool_calls:
                if verbose:
                    print(f"[step {step}] Final answer (no tool calls).")
                return response.content

            for call in tool_calls:
                if verbose:
                    print(f"\n[step {step}] Tool call -> {call['name']} args={call.get('args', {})}")
                tool_result = self._execute_tool(call)
                if verbose:
                    print(f"[step {step}] Tool result <-\n{tool_result}")

                self.messages.append(ToolMessage(content=tool_result, tool_call_id=call["id"]))

        return "Max steps reached. Try rephrasing or simplifying your request."

    def debug_state(self) -> Dict[str, Any]:
        return {"notes": list(self.notes), "num_messages": len(self.messages)}

agent = MinimalAgent(llm=llm, system_prompt=SYSTEM_PROMPT, tools=TOOLS)
print("Agent initialized ✅")


## 8) Test 1: Calculator (structured args)

This test demonstrates **structured tool input**:
- You ask for a calculation
- The model calls `calculator` with JSON args like `{"expression": "...", "precision": ...}`
- The agent executes it and returns a final answer


### What this cell does
- Runs a test prompt through the agent.
- Use these tests to verify tool calling, memory, and multi-step behavior.


In [ ]:
prompt = "Compute (18*7) + (2**10). Return the final number."
answer = agent.run(prompt, verbose=True)
print("\nAGENT OUTPUT:\n", answer)


## 9) Test 2: web_search (real)

This demonstrates tool selection for a lookup-style query.
We intentionally use a stub so the notebook works without extra services.

Later, you can replace `web_search` with Tavily/SerpAPI/your internal KB.


### What this cell does
- Runs a test prompt through the agent.
- Use these tests to verify tool calling, memory, and multi-step behavior.


In [ ]:
prompt = "Search for: top spain football matches today (top_k=2). Summarize in 2 bullet points."
answer = agent.run(prompt, verbose=True)
print("\nAGENT OUTPUT:\n", answer)


## 10) Test 3: Memory (remember + recall)

This demonstrates real memroy behavior.

We store a note using `remember(note=...)` and retrieve notes using `recall(limit=...)`.


### What this cell does
- Runs a test prompt through the agent.
- Use these tests to verify tool calling, memory, and multi-step behavior.


In [ ]:
prompt = "Remember that my project name is CoordiMap and I'm building a minimal agent demo."
answer = agent.run(prompt, verbose=True)
print("\nAGENT OUTPUT:\n", answer)

prompt2 = "Recall the last 5 notes."
answer2 = agent.run(prompt2, verbose=True)
print("\nAGENT OUTPUT:\n", answer2)


## 11) Test 4: Multi-step tool usage

This prompt encourages a sequence:
1) Search (tool)
2) Calculate (tool)
3) Remember (tool)
4) Synthesize final answer (no tool)

Watch the verbose output to see the tool calls and their results.


### What this cell does
- Runs a test prompt through the agent.
- Use these tests to verify tool calling, memory, and multi-step behavior.


In [ ]:
prompt = """
1) Search: Top LangChain agents news (top_k=2)
2) Compute: (12*12) with precision 0
3) Remember: "Agents follow a think-act-observe loop"
Then give me a short final explanation connecting the three.
"""

answer = agent.run(prompt, verbose=True)
print("\nAGENT OUTPUT:\n", answer)

print("\nDEBUG STATE:\n", agent.debug_state())


In [ ]:
prompt2 = "Recall the last 5 notes."
answer2 = agent.run(prompt2, verbose=True)
print("\nAGENT OUTPUT:\n", answer2)


##12) UI setup – install Gradio

We use Gradio to turn our agent into a simple interactive application.
This allows us to chat with the agent, see responses in real time, and make the demo feel like a real product rather than a script.

### What this cell does
- Installs the Python dependencies needed for the demo.
- If you restart the runtime, you may need to re-run this cell.
- In Colab, installs can take a while if the runtime is reconnecting.


In [ ]:
!pip -q install -U gradio

## 13) Expose tool activity for the UI

To make the agent’s behavior visible in the interface, we add a helper method that returns both the final answer and the sequence of tool calls and results.
This does not change how the agent works, it only exposes internal steps for display in the UI.

### What this cell does
- Adds `run_with_trace()` to capture **tool calls and tool results** for the UI.
- This does not change the agent logic, it only exposes a trace for debugging/teaching.


In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage
import types

def run_with_trace(self, user_input: str):
    """
    Like run(), but returns (final_answer, trace_events).
    trace_events is a list of dicts with tool call + tool result.
    """
    trace = []
    self.messages.append(HumanMessage(content=user_input))

    for step in range(self.max_steps):
        response = self.llm.bind_tools(self.tools).invoke(self.messages)
        self.messages.append(response)

        tool_calls = getattr(response, "tool_calls", None) or []
        if not tool_calls:
            return response.content, trace

        for call in tool_calls:
            trace.append({
                "type": "tool_call",
                "name": call["name"],
                "args": call.get("args", {})
            })

            tool_result = self._execute_tool(call)

            trace.append({
                "type": "tool_result",
                "name": call["name"],
                "result": tool_result
            })

            self.messages.append(ToolMessage(content=tool_result, tool_call_id=call["id"]))

    return "Max steps reached. Try rephrasing or simplifying your request.", trace

agent.run_with_trace = types.MethodType(run_with_trace, agent)
print("run_with_trace() attached ✅")


##14) Connect the agent to the UI

This function acts as the bridge between Gradio and the agent.
It receives user input from the chat interface, runs the agent, and returns the updated conversation along with a readable tool trace.

**Build the chat interface**

Here we define the Gradio layout: a chat window for the conversation and a separate panel that displays tool calls and results.
This dual view makes the demo both user-friendly and developer-friendly.

### What this cell does
- Adds `run_with_trace()` to capture **tool calls and tool results** for the UI.
- This does not change the agent logic, it only exposes a trace for debugging/teaching.


In [ ]:
import gradio as gr
import json
import traceback

def format_trace(trace):
    if not trace:
        return "No tool calls in this turn."
    lines = []
    for e in trace:
        if e["type"] == "tool_call":
            lines.append(
                f"🛠 TOOL CALL: {e['name']}\nargs: {json.dumps(e['args'], ensure_ascii=False)}\n"
            )
        elif e["type"] == "tool_result":
            lines.append(
                f"📦 TOOL RESULT: {e['name']}\n{e['result']}\n"
            )
    return "\n".join(lines)

def chat_fn(message, history):
    # In your Gradio build, history is expected to be a list of dict messages
    history = history or []

    try:
        answer, trace = agent.run_with_trace(message)

        # ✅ messages format: list of dicts with role + content
        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": answer},
        ]
        return history, format_trace(trace)

    except Exception:
        err = traceback.format_exc()
        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": "⚠️ Internal error. See Tool Trace for details."},
        ]
        return history, err

def clear_fn():
    global agent
    agent = MinimalAgent(llm=llm, system_prompt=SYSTEM_PROMPT, tools=TOOLS)

    # Reattach run_with_trace after reset
    import types
    agent.run_with_trace = types.MethodType(run_with_trace, agent)

    return [], "Cleared chat + agent memory."

with gr.Blocks(title="Minimal Agent (LangChain + Tools)") as demo:
    gr.Markdown(
        "# Minimal Agent (LangChain + OpenAI + Tools)\n\n"
        "**Try prompts like:**\n"
        "- `Search: LangChain agents top_k=3. Summarize in 3 bullets.`\n"
        "- `Compute (12*12) with precision 0`\n"
        "- `Remember that agents follow a think-act-observe loop`\n"
        "- `Recall the last 5 notes`"
    )

    # ✅ No `type=` argument (your version doesn't support it)
    chatbot = gr.Chatbot(height=420, label="Chat")
    trace_box = gr.Textbox(label="Tool Trace (Developer View)", lines=14, interactive=False)

    msg = gr.Textbox(placeholder="Type your message and press Enter...", label="Your message")

    with gr.Row():
        send = gr.Button("Send", variant="primary")
        clear = gr.Button("Clear", variant="secondary")

    send.click(chat_fn, inputs=[msg, chatbot], outputs=[chatbot, trace_box])
    msg.submit(chat_fn, inputs=[msg, chatbot], outputs=[chatbot, trace_box])
    clear.click(clear_fn, outputs=[chatbot, trace_box])



##15) Launch the agent application

Finally, we launch the Gradio app.
In Google Colab, this will generate a public link that participants can open in their browser to interact with the agent live.

### What this cell does
- Launches the Gradio app.
- In Colab, `share=True` produces a public link.
- Keep this cell running while you use the UI.


In [ ]:
demo.launch(share=True)

## 16) What to customize next (practical extensions)

Once you understand this notebook, you can evolve it into a “real” agent by:

- Replacing `web_search` with other real search tool (Bing Search, SerpAPI, or your internal KM)
- Adding a `query_sql(database, sql)` tool for structured data access
- Adding a `load_document(path)` tool to read PDFs/text for RAG-style workflows
- Adding citations: store sources alongside tool outputs and format them in the final answer
- Adding stricter tool validation and safety policy (allowed domains, allowed operations)
